In [9]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

True

Traceable decorator with the run type of LLM to render the information on LLM nicely.

In [10]:
from langsmith import traceable

inputs = [
  ["system", "You are a helpfull assistant."],
  ["user", "I'd like to book a table for two"]
]

output = {
  "choices": [
    {
      "messages": {
        "role": "assistant",
        "content": "Sure, what time would you like to book the table for?"
      }
    }
  ]
}

# Can also use one of:
# output = {
#   "message": {
#     "role": "assistant",
#     "content": "Sure, what time would you like to book the table for?"
#   }
# }
#
# output = {
#   "role": "assistant",
#   "content": "Sure, what time would you like to book the table for?"
# }
#
# output = ["assistant", "Sure, what time would you like to book the table for?"]

@traceable(
  run_type="llm"
)
def chat_model(messages: list):
  return output

chat_model(inputs)

{'choices': [{'messages': {'role': 'assistant',
    'content': 'Sure, what time would you like to book the table for?'}}]}

Retriever Runs + Document

In [11]:
from langchain_core.documents import Document
from typing import List

def _convert_docs(results) -> List[Document]:
  return [
    {
      # document type
      "page_content": r,
      "type": "Document",
      "metadata": {"foo": "bar"}
    }
    for r in results
  ]

@traceable(
    run_type="retriever"
)
def retrieve_langsmith_docs(query):
  # Retriever returning a hardcoded dummy documents.
  # In production, this could be a real vector database or other document index
  contents = ["LangSmith Document contents 1", "LangSmith Document contents 2", "LangSmith Document contents 3"]
  return _convert_docs(contents)

retrieve_langsmith_docs("Here should be a User Query")

[{'page_content': 'LangSmith Document contents 1',
  'type': 'Document',
  'metadata': {'foo': 'bar'}},
 {'page_content': 'LangSmith Document contents 2',
  'type': 'Document',
  'metadata': {'foo': 'bar'}},
 {'page_content': 'LangSmith Document contents 3',
  'type': 'Document',
  'metadata': {'foo': 'bar'}}]

Tool Calling

In [ ]:
from langsmith import traceable
from openai import OpenAI
from typing import List, Optional
from tavily import TavilyClient
import json

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
documents = []

@traceable(run_type="tool")
def search_web(query: str):
  web_docs = tavily_client.search(query = query, max_results=2)
  web_results = "\n".join([d["content"] for d in web_docs["results"]])
  return web_results

@traceable(run_type="llm")
def call_openai(messages: List[dict], tools: Optional[List[dict]]) -> str:
  return openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages = messages,
    temperature = 0,
    tools = tools
  )

@traceable(run_type="chain")
def respond(inputs, tools):
  response = call_openai(inputs, tools) # Used to figure out what tools to run.
  tool_call_args = json.loads(response.choices[0].message.tool_calls[0].function.arguments)
  query = tool_call_args["query"]
  tool_response_message = {
    "role": "tool",
    "content": json.dumps({
      "query": query,
      "results": search_web(query),
    }),
    "tool_call_id": response.choices[0].message.tool_calls[0].id
  }
  inputs.append(response.choices[0].message)
  inputs.append(tool_response_message)
  output = call_openai(inputs, None) # Used to read the results of those tools and write the final answer to the user.
  return output

tools = [
  {
    "type": "function",
    "function": {
      "name": "search_web",
      "description": "Search the web for a specific query",
      "parameters": {
        "type": "object",
        "properties": {
          "query": {
            "type": "string",
            "description": "The query to search the web for"
          }
        },
        "required": ["query"]
      }
    }
  }
]
inputs = [
  {"role": "system", "content": "You are a helpfull assistant"},
  {"role": "user", "content": "Who is Elon Mask?"},
]

respond(inputs, tools)

ChatCompletion(id='chatcmpl-EDA2xBrEOZ5n6FHKq9sZ7rtXSqq1N', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Elon Musk is an entrepreneur and business magnate known for founding and leading several high-profile technology companies. Born in 1971, he is the CEO and lead designer of SpaceX, CEO and product architect of Tesla, Inc., and has been involved in various other ventures, including Neuralink and The Boring Company.\n\nMusk is recognized for his efforts to address environmental, social, and economic challenges through innovation. He holds degrees in physics and economics from the University of Pennsylvania. His work has earned him numerous accolades, including being named one of TIME's 100 most influential people and receiving recognition as an innovator and entrepreneur of the year from various publications.\n\nIn addition to his business ventures, Musk is a trustee of the X Prize Foundation and has served on the boards of sever